# Blocco 2: Dal DataFrame a Excel

---

## La scena

Sono le 14:30. Il Dott. Ferretti bussa alla tua scrivania:

> *"Ottimo lavoro con i dati. Hai scaricato i prezzi, calcolato i rendimenti, tutto bene. Ma sai cosa manca? Il report. I Bianchi vengono venerdì pomeriggio e voglio un file Excel professionale sul loro tavolo — non un terminal Python. Inizia a costruirlo."*

In questo blocco impari a:

- **ES06** — Esportare un DataFrame pandas in Excel con xlwings
- **ES07** — Formattare il foglio con i colori e lo stile di Turin Wealth
- **ES08** — Inserire formule italiane da Python (e capire perché è diverso dall'inglese)
- **ES09** — Creare grafici professionali direttamente da Python

---

### Architettura xlwings: come funziona

```
Python  ──►  xlwings  ──►  COM/UNO  ──►  Excel (aperto)
              API            Bridge        Applicazione
```

xlwings non scrive file .xlsx direttamente: **parla con Excel attraverso COM** (Component Object Model su Windows). Questo significa che:

1. Excel deve essere **installato** sulla macchina
2. Excel viene **aperto** (visibile o in background)
3. Le operazioni avvengono **in tempo reale** nell'applicazione
4. Puoi vedere ogni modifica mentre il codice gira

Questo approccio ha un grande vantaggio rispetto a openpyxl o xlsxwriter: puoi usare **tutte le funzionalità di Excel**, incluse le formule in italiano, i grafici avanzati e la formattazione condizionale esattamente come farebbe un utente.

In [ ]:
import xlwings as xw
import pandas as pd
import numpy as np
import yfinance as yf
import sys, os

# Aggiungi la cartella scripts al path per importare i moduli Turin Wealth
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("")), "scripts"))
from tw_config import COLORS, AZIENDE, NUMBER_FORMATS
from tw_utils import set_formula, fmt_header, protect_sheet

# Directory per cache dati e output
CACHE_DIR = os.path.join(os.path.dirname(os.path.abspath("")), "lezione", "dati_cache")
OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath("")), "lezione")


def scarica_o_cache(ticker, period="5y"):
    """Scarica dati da Yahoo Finance, con fallback su cache locale."""
    try:
        df = yf.download(ticker, period=period, progress=False)
        if len(df) > 0:
            return df
    except Exception:
        pass
    # Fallback: leggi dalla cache locale
    nome_file = ticker.replace("^", "").replace(".", "_") + ".csv"
    path_cache = os.path.join(CACHE_DIR, nome_file)
    if os.path.exists(path_cache):
        print(f"Uso cache locale per {ticker}")
        return pd.read_csv(path_cache, index_col=0, parse_dates=True)
    raise FileNotFoundError(f"Nessun dato per {ticker}")


# Carica dati azioni (riprende dal Blocco 1)
tickers = [az["ticker"] for az in AZIENDE]
nomi = [az["nome"] for az in AZIENDE]

dati_list = []
for t in tickers:
    df = scarica_o_cache(t)
    dati_list.append(df["Close"].rename(t))

prezzi = pd.concat(dati_list, axis=1)
prezzi.columns = nomi

print(f"Dati caricati: {len(prezzi)} giorni x {len(prezzi.columns)} titoli")
print(f"Periodo: {prezzi.index[0].date()} -> {prezzi.index[-1].date()}")
prezzi.tail(3)

### Perché xlwings e non openpyxl?

| Libreria | Come scrive | Formule | Grafici | Formattazione |
|---|---|---|---|---|
| **openpyxl** | File XML diretto | Limitata | Limitati | Media |
| **xlsxwriter** | File XML diretto | No (solo testo) | Buoni | Buona |
| **xlwings** | Via COM con Excel | Completa (IT/EN) | Completi | Completa |

Per questo corso usiamo xlwings perché replica esattamente quello che farebbe un utente Excel — incluse le **formule in italiano** con punto e virgola come separatore.

> **Requisito**: Excel deve essere installato. Su Mac funziona con Excel per Mac. Su Linux non funziona (usa openpyxl come alternativa).

---
## ES06: Da pandas a Excel

> *"Il primo passo: portare i dati in Excel"*

Ferretti ti guarda: *"Un DataFrame pandas è utile per te, ma inutile per i Bianchi. Devo vederlo in Excel, con date, intestazioni, tutto formattato."*

Il pattern fondamentale di xlwings è semplice:

```python
ws.range("A1").value = dataframe       # Scrivi
ws.range("A1").options(pd.DataFrame, expand="table").value  # Leggi
```

Il parametro `.options(index=True, header=True)` controlla se scrivere anche l'indice e le intestazioni delle colonne.

In [ ]:
# Apri Excel (visible=True: lo vediamo mentre il codice gira)
app = xw.App(visible=True)
wb = app.books.add()
ws = wb.sheets[0]
ws.name = "Prezzi"

# Titolo principale
ws.range("A1").value = "QUOTAZIONI STORICHE - Turin Wealth Advisory"
ws.range("A1").font.size = 14
ws.range("A1").font.bold = True

# Scrivi il DataFrame a partire da A3
# options(index=True): include la colonna delle date
# options(header=True): include i nomi delle colonne come prima riga
ultimi_60 = prezzi.tail(60)
ws.range("A3").options(index=True, header=True).value = ultimi_60

print(f"Scritte {len(ultimi_60)} righe x {len(ultimi_60.columns)} colonne")
print(f"Range occupato: A3 -> {chr(65 + len(ultimi_60.columns))}{ 3 + len(ultimi_60)}")

### Cosa ha fatto xlwings?

Guardando Excel, dovresti vedere:
- **A3**: intestazione `Date` (indice del DataFrame)
- **B3:F3**: nomi delle colonne (Terna, Ferrari, Microsoft, Alphabet, LVMH)
- **A4:A63**: date in formato Excel
- **B4:F63**: valori numerici dei prezzi

xlwings converte automaticamente:
- `datetime` di pandas → date Excel
- `float64` → numeri Excel
- `NaN` → celle vuote
- `str` → testo Excel

Puoi anche **leggere indietro** da Excel a pandas:

In [ ]:
# Leggi i dati indietro da Excel a pandas
# expand="table" trova automaticamente i bordi del range
dati_letti = ws.range("A3").options(pd.DataFrame, expand="table").value

print(f"Tipo ritornato: {type(dati_letti)}")
print(f"Shape: {dati_letti.shape}")
dati_letti.head()

In [ ]:
# ============================================================
# ESERCIZIO ES06
# ============================================================
# Crea un secondo foglio "Rendimenti" con i rendimenti giornalieri
# I rendimenti percentuali si calcolano come:
#   (prezzo_oggi - prezzo_ieri) / prezzo_ieri
# In pandas: df.pct_change()
#
# Obiettivo:
# 1. Calcola i rendimenti giornalieri dal DataFrame prezzi
# 2. Aggiungi un nuovo foglio al workbook e chiamalo "Rendimenti"
# 3. Scrivi il titolo "RENDIMENTI GIORNALIERI - Turin Wealth Advisory" in A1
# 4. Scrivi i rendimenti a partire dalla cella A3 (con indice e intestazioni)
# ============================================================

rendimenti = # ??? (calcola i rendimenti percentuali giornalieri da prezzi)

ws_rend = # ??? (aggiungi un nuovo foglio al workbook wb)
ws_rend.name = # ??? (assegna il nome "Rendimenti")

# Scrivi il titolo in A1
# ???

# Scrivi i rendimenti a partire da A3 (con index e header)
# ???

print("Foglio Rendimenti creato!")

In [ ]:
# ============================================================
# SOLUZIONE ES06
# ============================================================

# 1. Calcola i rendimenti percentuali giornalieri
rendimenti = prezzi.pct_change().dropna()

# 2. Aggiungi nuovo foglio
ws_rend = wb.sheets.add("Rendimenti")

# 3. Titolo
ws_rend.range("A1").value = "RENDIMENTI GIORNALIERI - Turin Wealth Advisory"
ws_rend.range("A1").font.size = 14
ws_rend.range("A1").font.bold = True

# 4. Scrivi i rendimenti
ws_rend.range("A3").options(index=True, header=True).value = rendimenti

print(f"Foglio Rendimenti creato: {len(rendimenti)} righe x {len(rendimenti.columns)} colonne")
print(f"Periodo: {rendimenti.index[0].date()} -> {rendimenti.index[-1].date()}")

---
## ES07: Formattazione professionale

> *Ferretti: "Un report senza formattazione non è un report. È un dump di dati. I Bianchi pagano per qualcosa di professionale."*

La formattazione in xlwings segue la stessa logica di Excel:
- `.color` → sfondo cella (RGB tuple)
- `.font.color` → colore testo
- `.font.bold` → grassetto
- `.api.NumberFormatLocal` → formato numero (versione italiana!)

### ATTENZIONE CRITICA: NumberFormatLocal vs number_format

Questa è la trappola più comune su Excel italiano:

```python
# SBAGLIATO su Excel italiano:
rng.number_format = "0.0%"     # Il . viene letto come SEPARATORE MIGLIAIA!
                                # Risultato: "04%" invece di "3,5%"

# SBAGLIATO su Excel italiano:
rng.number_format = "#,##0"    # La , viene letta come DECIMALE!
                                # Risultato: "3000000,0" invece di "3.000.000"

# CORRETTO su Excel italiano:
rng.api.NumberFormatLocal = "#.##0,00 €"   # . = migliaia, , = decimale
rng.api.NumberFormatLocal = "0,00%"        # percentuale con 2 decimali
rng.api.NumberFormatLocal = "+0,0%;-0,0%"  # con segno
```

La regola: **usare sempre `api.NumberFormatLocal`** con la sintassi italiana (punto per migliaia, virgola per decimali).

In [ ]:
# Torna al foglio Prezzi
ws = wb.sheets["Prezzi"]

# Colori dal tema Turin Wealth (da tw_config.py)
HEADER_COLOR = COLORS["header"]       # (44, 62, 80)  - blu profondo
ACCENT = COLORS["accent"]             # (231, 76, 60) - corallo energetico
GOLD = COLORS["gold"]                 # (243, 156, 18) - ambra
WHITE = COLORS["header_text"]         # (255, 255, 255) - bianco
ALT_ROW = COLORS["table_alt_row"]     # (248, 249, 250) - grigio chiaro

print("Colori caricati da tw_config.py:")
print(f"  Header: RGB{HEADER_COLOR}")
print(f"  Accent: RGB{ACCENT}")
print(f"  Gold:   RGB{GOLD}")

# Formatta riga di intestazione (riga 3)
header_range = ws.range("A3").expand("right")
header_range.color = HEADER_COLOR
header_range.font.color = WHITE
header_range.font.bold = True
header_range.font.size = 11

# Formato numeri - ATTENZIONE: Excel italiano!
# Calcola il range dati (dalla riga 4 fino all'ultima)
ultima_riga = ws.range("A3").expand("down").last_cell.row
data_range = ws.range(f"B4:F{ultima_riga}")
data_range.api.NumberFormatLocal = "#.##0,00 €"  # . = migliaia, , = decimale

# Righe alternate per leggibilità
for i in range(4, ultima_riga + 1):
    if i % 2 == 0:  # Righe pari
        ws.range(f"A{i}:F{i}").color = ALT_ROW

# Autofit colonne (adatta larghezza al contenuto)
ws.autofit()

print(f"Formattazione applicata a {ultima_riga - 3} righe di dati")

### Il dizionario COLORS

`tw_config.py` contiene la palette completa di Turin Wealth:

```python
COLORS["header"]         # (44, 62, 80)   - blu profondo (header tabelle)
COLORS["accent"]         # (231, 76, 60)  - corallo (accenti, alert)
COLORS["gold"]           # (243, 156, 18) - ambra (stelle, elementi premium)
COLORS["table_alt_row"]  # (248, 249, 250) - grigio chiaro (righe alternate)
COLORS["correct_bg"]     # (212, 239, 223) - verde (valori positivi)
COLORS["wrong_bg"]       # (250, 219, 216) - rosso (valori negativi)
COLORS["chart_primary"]  # (44, 62, 80)   - blu per grafici
```

Usare sempre questi colori per coerenza con gli altri file del corso.

In [ ]:
# Bordi professionali con xlwings API
# I border_id corrispondono alle posizioni: 7=sinistra, 8=destra, 9=top, 10=bottom

table = ws.range(f"A3:F{ultima_riga}")

# Converti colore bordo in intero BGR (formato Excel interno)
border_color = COLORS["border_gray"]
border_color_int = border_color[0] + border_color[1] * 256 + border_color[2] * 65536

for border_id in [7, 8, 9, 10]:  # Left, Right, Top, Bottom
    border = table.api.Borders(border_id)
    border.LineStyle = 1  # xlContinuous
    border.Weight = 2     # xlThin
    border.Color = border_color_int

print("Bordi applicati alla tabella Prezzi")

In [ ]:
# ============================================================
# ESERCIZIO ES07
# ============================================================
# Formatta il foglio "Rendimenti" in modo professionale
#
# Obiettivo:
# 1. Applica lo stesso header colorato (HEADER_COLOR + WHITE) alla riga 3
# 2. Formato percentuale per i rendimenti: "0,00%" (2 decimali)
#    Ricorda: usa api.NumberFormatLocal, NON number_format!
# 3. Aggiungi formattazione condizionale: celle con rendimento > 0 in verde,
#    celle con rendimento < 0 in rosso (usa COLORS["correct_bg"] e COLORS["wrong_bg"])
#    Suggerimento: puoi usare un ciclo for sulle celle e confrontare .value
# ============================================================

ws_rend = wb.sheets["Rendimenti"]
ultima_riga_rend = ws_rend.range("A3").expand("down").last_cell.row

# 1. Formatta header riga 3
# ???

# 2. Formato percentuale per i dati (dalla riga 4)
# ???

# 3. Colora le celle: verde se > 0, rosso se < 0
#    Suggerimento: leggi i valori con ws_rend.range(...).value
#    poi applica .color cella per cella
# ???

ws_rend.autofit()
print("Foglio Rendimenti formattato!")

In [ ]:
# ============================================================
# SOLUZIONE ES07
# ============================================================

ws_rend = wb.sheets["Rendimenti"]
ultima_riga_rend = ws_rend.range("A3").expand("down").last_cell.row

# 1. Formatta header riga 3
header_rend = ws_rend.range("A3").expand("right")
header_rend.color = COLORS["header"]
header_rend.font.color = COLORS["header_text"]
header_rend.font.bold = True
header_rend.font.size = 11

# 2. Formato percentuale per i dati
dati_rend = ws_rend.range(f"B4:F{ultima_riga_rend}")
dati_rend.api.NumberFormatLocal = "0,00%"  # Italiano: virgola = decimale

# 3. Colora celle: verde se positivo, rosso se negativo
verde = COLORS["correct_bg"]   # (212, 239, 223)
rosso = COLORS["wrong_bg"]     # (250, 219, 216)

valori = dati_rend.value  # Legge tutta la matrice in una sola chiamata (piu veloce)
for r_idx, riga in enumerate(valori):
    for c_idx, val in enumerate(riga):
        if val is not None and isinstance(val, (int, float)):
            cella = ws_rend.range(f"{chr(66 + c_idx)}{4 + r_idx}")
            cella.color = verde if val > 0 else rosso

ws_rend.autofit()
print(f"Formattazione completata: {ultima_riga_rend - 3} righe, colori verde/rosso applicati")

---
## ES08: Formule italiane da Python

> *"Il segreto di Turin Wealth: formule Excel scritte da Python"*

Ferretti ti spiega: *"Non basta copiare i numeri. Il file Excel deve contenere FORMULE reali — così quando arrivano nuovi dati, il report si aggiorna da solo."*

### Il problema dell'operatore `@`

Excel 365 ha introdotto le **dynamic array formulas**. Quando usi metodi "legacy" per impostare formule, Excel 365 aggiunge automaticamente `@` davanti:

```excel
=@MEDIA(B4:F4)    # Quello che vedi in Excel 365 (sbagliato!)
=MEDIA(B4:F4)     # Quello che volevi (corretto)
```

L'operatore `@` forza la **implicit intersection** (comportamento legacy). Per le nostre formule è inutile e confonde gli studenti.

### La soluzione: `set_formula()` da tw_utils.py

```python
# SBAGLIATO - aggiunge @ in Excel 365
ws.range("B5").api.FormulaLocal = "=MEDIA(B4:F4)"

# CORRETTO - niente @!
from tw_utils import set_formula
set_formula(ws.range("B5"), "=MEDIA(B4:F4)")
```

Internamente `set_formula()` usa `Formula2Local` (dynamic array API) invece di `FormulaLocal`:

```python
def set_formula(rng, formula_it):
    """Imposta una formula italiana senza @ in Excel 365."""
    try:
        rng.api.Formula2Local = formula_it  # Dynamic array - no @
    except AttributeError:
        rng.api.FormulaLocal = formula_it   # Fallback versioni precedenti
```

### Sintassi italiana: punto e virgola!

```excel
# ITALIANO (Excel IT)       # INGLESE (Excel EN)
=SE(A1>10;"Grande";"Piccolo")    =IF(A1>10,"Grande","Piccolo")
=MEDIA(B4:F4)                    =AVERAGE(B4:F4)
=CONTA.SE(A:A;">0")              =COUNTIF(A:A,">0")
=CERCA.VERT(X;A:B;2;FALSO)       =VLOOKUP(X,A:B,2,FALSE)
```

Il separatore degli argomenti in Excel italiano è **`;`** non **`,`**.

In [ ]:
# Dimostrazione del problema @ e della soluzione set_formula()
ws = wb.sheets["Prezzi"]
ultima_riga = ws.range("A3").expand("down").last_cell.row

# SBAGLIATO - FormulaLocal aggiunge @ in Excel 365
ws.range("G3").value = "Media (con @)"
ws.range("G3").font.bold = True
ws.range("G3").color = COLORS["wrong_bg"]
ws.range("G4").api.FormulaLocal = "=MEDIA(B4:F4)"  # Excel aggiunge @!

import time
time.sleep(1)  # Pausa per vedere il risultato

# Leggi cosa ha scritto Excel effettivamente
formula_scritta = ws.range("G4").api.FormulaLocal
print(f"Formula scritta da FormulaLocal: '{formula_scritta}'")
print(f"Ha aggiunto @? {'Si!' if '@' in formula_scritta else 'No'}")

# CORRETTO - set_formula() usa Formula2Local
ws.range("H3").value = "Media (corretta)"
ws.range("H3").font.bold = True
ws.range("H3").color = COLORS["correct_bg"]
set_formula(ws.range("H4"), "=MEDIA(B4:F4)")

time.sleep(1)
formula_corretta = ws.range("H4").api.FormulaLocal
print(f"Formula scritta da set_formula(): '{formula_corretta}'")
print(f"Ha aggiunto @? {'Si!' if '@' in formula_corretta else 'No'}")

### Formula2Local: il dettaglio tecnico

Excel 365 introduce due API parallele per le formule:

| API | Comportamento | Uso consigliato |
|---|---|---|
| `FormulaLocal` | Legacy, aggiunge `@` per formule non-array | Da evitare |
| `Formula2Local` | Dynamic array, niente `@` | Usare sempre |

La funzione `set_formula()` di `tw_utils.py` gestisce entrambi i casi con try/except, garantendo compatibilità anche con versioni Excel precedenti alla 365.

**Regola pratica**: usa sempre `set_formula()` e non preoccuparti dei dettagli interni.

In [ ]:
# Aggiunge colonne calcolate con formule italiane
ws = wb.sheets["Prezzi"]
ultima_riga = ws.range("A3").expand("down").last_cell.row

# Rimuovi le colonne G (con @) e H (demo), usa G come Media definitiva
ws.range(f"G3:H{ultima_riga}").clear()

# Intestazioni per le colonne calcolate
ws.range("G3").value = "Media"
ws.range("H3").value = "Min"
ws.range("I3").value = "Max"

# Formatta le intestazioni
fmt_header(ws.range("G3:I3"))  # Funzione da tw_utils.py

# Scrivi le formule per ogni riga di dati
# NOTA: il separatore e il PUNTO E VIRGOLA (;) non la virgola!
for row in range(4, ultima_riga + 1):
    set_formula(ws.range(f"G{row}"), f"=MEDIA(B{row}:F{row})")
    set_formula(ws.range(f"H{row}"), f"=MIN(B{row}:F{row})")
    set_formula(ws.range(f"I{row}"), f"=MAX(B{row}:F{row})")

# Formato numeri per le nuove colonne
ws.range(f"G4:I{ultima_riga}").api.NumberFormatLocal = "#.##0,00 €"

ws.autofit()
print(f"Aggiunte 3 colonne calcolate (righe 4-{ultima_riga}) con formule MEDIA/MIN/MAX")
print("Esempio formula in G4:", ws.range("G4").api.FormulaLocal)

In [ ]:
# ============================================================
# ESERCIZIO ES08
# ============================================================
# Aggiungi colonne calcolate al foglio "Rendimenti"
#
# Obiettivo:
# 1. Aggiungi una colonna "Positivi" che conta quanti rendimenti > 0
#    nella riga (cioè quanti titoli sono saliti quel giorno)
#    Funzione italiana: CONTA.SE con criterio ">0"
#    RICORDA: separatore = punto e virgola (;)
#    Esempio struttura: =CONTA.SE(B4:F4;">0")
#
# 2. Aggiungi una colonna "Media" con la media dei rendimenti della riga
#    Funzione italiana: MEDIA
#
# IMPORTANTE: usa set_formula() per entrambe le colonne!
# ============================================================

ws_rend = wb.sheets["Rendimenti"]
ultima_riga_rend = ws_rend.range("A3").expand("down").last_cell.row

# Intestazioni nuove colonne (mettile in G3 e H3)
ws_rend.range("G3").value = "Positivi"
ws_rend.range("H3").value = "Media"
fmt_header(ws_rend.range("G3:H3"))

# 1. Colonna Positivi: CONTA.SE con criterio ">0"
# Suggerimento: separatore italiano = ; (punto e virgola)
for row in range(4, ultima_riga_rend + 1):
    set_formula(ws_rend.range(f"G{row}"), # ??? formula CONTA.SE
    )

# 2. Colonna Media: MEDIA dei rendimenti
for row in range(4, ultima_riga_rend + 1):
    set_formula(ws_rend.range(f"H{row}"), # ??? formula MEDIA
    )

# Formato per le nuove colonne
ws_rend.range(f"G4:G{ultima_riga_rend}").api.NumberFormatLocal = "0"  # Intero (conteggio)
ws_rend.range(f"H4:H{ultima_riga_rend}").api.NumberFormatLocal = "0,00%"  # Percentuale

ws_rend.autofit()
print("Colonne calcolate aggiunte al foglio Rendimenti!")

In [ ]:
# ============================================================
# SOLUZIONE ES08
# ============================================================

ws_rend = wb.sheets["Rendimenti"]
ultima_riga_rend = ws_rend.range("A3").expand("down").last_cell.row

# Intestazioni
ws_rend.range("G3").value = "Positivi"
ws_rend.range("H3").value = "Media"
fmt_header(ws_rend.range("G3:H3"))

# 1. Colonna Positivi: CONTA.SE (separatore italiano = ;)
for row in range(4, ultima_riga_rend + 1):
    set_formula(ws_rend.range(f"G{row}"), f"=CONTA.SE(B{row}:F{row};\">0\")")

# 2. Colonna Media
for row in range(4, ultima_riga_rend + 1):
    set_formula(ws_rend.range(f"H{row}"), f"=MEDIA(B{row}:F{row})")

# Formati
ws_rend.range(f"G4:G{ultima_riga_rend}").api.NumberFormatLocal = "0"
ws_rend.range(f"H4:H{ultima_riga_rend}").api.NumberFormatLocal = "0,00%"

ws_rend.autofit()

# Verifica: mostra la formula scritta in G4
formula_g4 = ws_rend.range("G4").api.FormulaLocal
formula_h4 = ws_rend.range("H4").api.FormulaLocal
print(f"Formula G4 (Positivi): {formula_g4}")
print(f"Formula H4 (Media):    {formula_h4}")
ha_chiocciola = "@" in formula_g4 or "@" in formula_h4
print(f"Presenza @: {'SI (problema!)' if ha_chiocciola else 'NO (corretto!)'}")

---
## ES09: Il grafico che parla

> *Ferretti: "I Bianchi non vogliono tabelle di numeri. Vogliono GRAFICI. Uno sguardo e capiscono tutto."*

xlwings permette di creare grafici completi direttamente da Python, usando la stessa API di Excel VBA.

### Tipi di grafico disponibili

```python
chart.chart_type = "line"     # Linee (andamento nel tempo)
chart.chart_type = "bar"      # Barre orizzontali (confronto)
chart.chart_type = "column"   # Istogramma verticale
chart.chart_type = "pie"      # Torta (composizione)
chart.chart_type = "area"     # Area (tendenza cumulativa)
chart.chart_type = "scatter"  # Dispersione (correlazione)
```

### L'accessor `api[1]`

Per personalizzazioni avanzate (titoli, colori serie, assi), si accede all'oggetto Chart nativo di Excel tramite `chart.api[1]`. Questo è l'equivalente di `ActiveChart` in VBA.

In [ ]:
ws = wb.sheets["Prezzi"]
ultima_riga = ws.range("A3").expand("down").last_cell.row

# Posizione del grafico: sotto i dati
# Usiamo le coordinate pixel di Excel (left, top in punti)
top_grafico = ws.range(f"A{ultima_riga + 3}").top

# Crea grafico a linee per l'andamento prezzi
chart = ws.charts.add(
    left=ws.range("A1").left,
    top=top_grafico,
    width=650,
    height=370
)
chart.chart_type = "line"

# Imposta i dati sorgente (solo colonne A:F, le prime 6 = date + 5 titoli)
chart.set_source_data(ws.range(f"A3:F{ultima_riga}"))

# Titolo del grafico
chart.api[1].HasTitle = True
chart.api[1].ChartTitle.Text = "Andamento Prezzi - 5 Titoli Turin Wealth"
chart.api[1].ChartTitle.Font.Size = 13
chart.api[1].ChartTitle.Font.Bold = True

# Personalizza colori delle serie (uno per titolo)
colori_grafico = [
    COLORS["chart_primary"],    # Blu profondo - Terna
    COLORS["chart_secondary"],  # Blu chiaro - Ferrari
    COLORS["chart_accent1"],    # Verde acqua - Microsoft
    COLORS["chart_accent2"],    # Ambra - Alphabet
    COLORS["chart_accent3"],    # Corallo - LVMH
]

for i, colore in enumerate(colori_grafico):
    try:
        serie = chart.api[1].SeriesCollection(i + 1)  # 1-indexed in Excel
        # Converti RGB in intero BGR
        colore_int = colore[0] + colore[1] * 256 + colore[2] * 65536
        serie.Format.Line.ForeColor.RGB = colore_int
        serie.Format.Line.Weight = 2  # Linea leggermente più spessa
    except Exception:
        pass  # Ignora se la serie non esiste

print(f"Grafico linee creato sotto i dati (riga {ultima_riga + 3})")

### Perché `api[1]`?

In xlwings, `chart.api` restituisce una tupla con due oggetti COM:
- `chart.api[0]` → `ChartObject` (il contenitore/cornice nel foglio)
- `chart.api[1]` → `Chart` (il grafico vero e proprio, con assi, serie, titoli)

Per modificare il contenuto del grafico (titoli, serie, colori), si usa sempre `chart.api[1]`.

Questa è la stessa API usata in VBA con `ActiveChart`, quindi qualsiasi codice VBA per grafici si traduce direttamente.

In [ ]:
# Grafico a torta - distribuzione per ultimo prezzo
ws_alloc = wb.sheets.add("Allocazione")

# Prepara dati per il grafico a torta
nomi_az = list(prezzi.columns)
ultimi_prezzi = prezzi.iloc[-1].values

# Scrivi i dati nel foglio
ws_alloc.range("A1").value = "Azienda"
ws_alloc.range("B1").value = "Ultimo Prezzo"
ws_alloc.range("A1:B1").font.bold = True
ws_alloc.range("A1:B1").color = COLORS["header"]
ws_alloc.range("A1:B1").font.color = COLORS["header_text"]

for i, (nome, prezzo) in enumerate(zip(nomi_az, ultimi_prezzi)):
    ws_alloc.range(f"A{i+2}").value = nome
    ws_alloc.range(f"B{i+2}").value = round(prezzo, 2)

ws_alloc.range("B2:B6").api.NumberFormatLocal = "#.##0,00 €"
ws_alloc.autofit()

# Crea grafico a torta
chart_pie = ws_alloc.charts.add(left=220, top=10, width=420, height=320)
chart_pie.chart_type = "pie"
chart_pie.set_source_data(ws_alloc.range(f"A1:B{len(nomi_az)+1}"))

chart_pie.api[1].HasTitle = True
chart_pie.api[1].ChartTitle.Text = "Distribuzione per Titolo (ultimo prezzo)"
chart_pie.api[1].ChartTitle.Font.Size = 12

# Mostra percentuali sulle fette
try:
    chart_pie.api[1].SeriesCollection(1).DataLabels().ShowPercentage = True
    chart_pie.api[1].SeriesCollection(1).DataLabels().ShowValue = False
except Exception:
    pass

print("Grafico a torta creato nel foglio Allocazione")
print("Titoli:", nomi_az)
print("Ultimi prezzi:", [f"{p:.2f}" for p in ultimi_prezzi])

In [ ]:
# ============================================================
# ESERCIZIO ES09
# ============================================================
# Crea un grafico a barre con i rendimenti annualizzati
#
# Obiettivo:
# 1. Crea un nuovo foglio chiamato "Confronto"
# 2. Calcola il rendimento annualizzato per ogni titolo:
#    rendimento_totale = (ultimo_prezzo / primo_prezzo) - 1
#    anni = len(prezzi) / 252  (252 giorni di borsa per anno)
#    rendimento_annualizzato = (1 + rendimento_totale) ** (1/anni) - 1
# 3. Scrivi nomi e rendimenti annualizzati nel foglio
# 4. Crea un grafico a barre (chart_type = "bar")
#    con titolo "Rendimento Annualizzato - Turin Wealth Portfolio"
# ============================================================

# 1. Crea nuovo foglio
ws_conf = # ???
ws_conf.name = # ???

# 2. Calcola i rendimenti annualizzati
anni = len(prezzi) / 252
rend_annualizzati = []
for nome in nomi_az:
    primo_prezzo = # ??? (primo prezzo disponibile per questo titolo)
    ultimo_prezzo = # ??? (ultimo prezzo disponibile)
    rend_totale = # ??? (variazione percentuale totale)
    rend_ann = # ??? (annualizzato con formula sopra)
    rend_annualizzati.append(rend_ann)

# 3. Scrivi i dati nel foglio
ws_conf.range("A1").value = "Titolo"
ws_conf.range("B1").value = "Rendimento Annualizzato"
# ???

# 4. Crea grafico a barre
chart_bar = # ???
chart_bar.chart_type = "bar"  # barre orizzontali
# ???

print("Grafico confronto creato!")
for nome, rend in zip(nomi_az, rend_annualizzati):
    print(f"  {nome}: {rend:.1%}")

In [ ]:
# ============================================================
# SOLUZIONE ES09
# ============================================================

# 1. Crea nuovo foglio
ws_conf = wb.sheets.add("Confronto")

# 2. Calcola i rendimenti annualizzati
anni = len(prezzi) / 252
nomi_az = list(prezzi.columns)
rend_annualizzati = []

for nome in nomi_az:
    serie = prezzi[nome].dropna()
    primo_prezzo = serie.iloc[0]
    ultimo_prezzo = serie.iloc[-1]
    rend_totale = (ultimo_prezzo / primo_prezzo) - 1
    rend_ann = (1 + rend_totale) ** (1 / anni) - 1
    rend_annualizzati.append(rend_ann)

# 3. Scrivi i dati nel foglio
ws_conf.range("A1").value = "Titolo"
ws_conf.range("B1").value = "Rendimento Annualizzato"
ws_conf.range("A1:B1").font.bold = True
ws_conf.range("A1:B1").color = COLORS["header"]
ws_conf.range("A1:B1").font.color = COLORS["header_text"]

for i, (nome, rend) in enumerate(zip(nomi_az, rend_annualizzati)):
    ws_conf.range(f"A{i+2}").value = nome
    ws_conf.range(f"B{i+2}").value = rend

# Formato percentuale per la colonna rendimenti
ws_conf.range(f"B2:B{len(nomi_az)+1}").api.NumberFormatLocal = "0,0%"
ws_conf.autofit()

# 4. Crea grafico a barre
chart_bar = ws_conf.charts.add(left=220, top=10, width=450, height=300)
chart_bar.chart_type = "bar"
chart_bar.set_source_data(ws_conf.range(f"A1:B{len(nomi_az)+1}"))

chart_bar.api[1].HasTitle = True
chart_bar.api[1].ChartTitle.Text = "Rendimento Annualizzato - Turin Wealth Portfolio"
chart_bar.api[1].ChartTitle.Font.Size = 12

# Colora le barre positive in verde, negative in rosso
try:
    serie_bar = chart_bar.api[1].SeriesCollection(1)
    for i, rend in enumerate(rend_annualizzati):
        colore = COLORS["chart_accent1"] if rend > 0 else COLORS["chart_accent3"]
        colore_int = colore[0] + colore[1] * 256 + colore[2] * 65536
        serie_bar.Points(i + 1).Format.Fill.ForeColor.RGB = colore_int
except Exception:
    pass

print("Grafico confronto creato!")
for nome, rend in zip(nomi_az, rend_annualizzati):
    segno = "+" if rend > 0 else ""
    print(f"  {nome}: {segno}{rend:.1%}")

In [ ]:
# Salva il file di output
output_path = os.path.join(OUTPUT_DIR, "report_blocco2.xlsx")
wb.save(output_path)
print(f"File salvato: {output_path}")
print("Fogli presenti nel workbook:")
for s in wb.sheets:
    print(f"  - {s.name}")
print()
print("NON chiudiamo Excel: il file verra usato nel Blocco 3!")
# wb.close()
# app.quit()

---
## Riepilogo Blocco 2

In questo blocco hai imparato a:

| Esercizio | Concetto chiave | Funzione/Metodo |
|---|---|---|
| **ES06** | Esportare DataFrame in Excel | `ws.range().options(index=True, header=True).value = df` |
| **ES06** | Leggere da Excel in pandas | `.options(pd.DataFrame, expand="table").value` |
| **ES07** | Formattare con colori Turin Wealth | `.color`, `.font.color`, `.font.bold` |
| **ES07** | Formato numeri italiano | `.api.NumberFormatLocal = "#.##0,00 €"` |
| **ES08** | Scrivere formule senza `@` | `set_formula(rng, "=MEDIA(B4:F4)")` |
| **ES08** | Sintassi italiana Excel | Separatore `;` non `,` |
| **ES09** | Creare grafici da Python | `ws.charts.add(...)`, `chart.chart_type` |
| **ES09** | Personalizzare serie grafici | `chart.api[1].SeriesCollection(i).Format.Line` |

### Regole da ricordare

1. **NumberFormatLocal** con sintassi italiana (`.` = migliaia, `,` = decimale)
2. **set_formula()** invece di `.api.FormulaLocal` per evitare `@`
3. **Punto e virgola** (`;`) come separatore nelle formule italiane
4. **`chart.api[1]`** per accedere al Chart nativo e personalizzarlo
5. **COLORS** da `tw_config.py` per coerenza visiva con tutto il progetto

---

### Blocco 3: Report multi-foglio e protezione

Nel prossimo blocco costruirai un report completo su piu fogli con:
- **Dashboard** riepilogativa con KPI
- **Protezione** celle per impedire modifiche accidentali
- **Validazione dati** con liste a tendina
- **Formule tra fogli** con riferimenti cross-sheet
- **Export PDF** per la presentazione ai Bianchi

> *Ferretti: "Ottimo lavoro. Ma questo e solo il primo report. Venerdi i Bianchi vogliono qualcosa di piu strutturato — una dashboard con tutti i KPI in una pagina. Ci vediamo domani mattina."*